In [ ]:
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

In [ ]:
import sys
if IN_COLAB:
    !git clone -q https://github.com/lukaslaobeyer/token-opt.git
    !pip install -q --progress-bar off jaxtyping open_clip_torch omegaconf
    sys.path.insert(0, "token-opt")
else:
    sys.path.insert(0, "..")

In [ ]:
import os
# Set this environment for deterministic execution
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

In [ ]:
import torch
# Enable for deterministic algorithms
torch.use_deterministic_algorithms(True, warn_only=False)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

from pathlib import Path
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as v2
import torchvision.transforms.v2.functional as tvf
from torchvision.datasets import ImageNet
from einops import rearrange

In [ ]:
from tto.test_time_opt import (
    TestTimeOpt,
    TestTimeOptConfig,
    CLIPObjective,
)

In [ ]:
#config variables
gpus = "8,9"

In [ ]:
device = torch.device("cuda")

## Utils

In [ ]:
def load_img(path, device=None):
    if IN_COLAB:
        path = "./token-opt/notebooks" / Path(path)
    img = (1. / 255.) * torch.from_numpy(
        np.array(Image.open(path)).astype(np.float32)
    ).permute(2, 0, 1)
    img = tvf.resize(img, 256)
    img = tvf.center_crop(img, 256)
    img = img.unsqueeze(0)
    if device is not None:
        img = img.to(device)
    return img

def display_image(*tensors):
    tensors = [255. * t.squeeze() for t in tensors]
    img = Image.fromarray(rearrange(
        tensors, "b c h w -> h (b w) c"
    ).to("cpu", dtype=torch.uint8).numpy())
    display(img)

def opt_callback(info):
    if info.i % 50 == 0:
        print(f"i = {info.i}")
        print("  CLIP score =", "\t".join(
            map(lambda l: f"{-l:.3f}", info.loss))
        )
        imgs = tto.decode(info.tokens).clamp(0., 1.)
        display_image(*imgs)

# Set up the objective function

In [ ]:
# Use CLIP similarity maximization objective
objective = CLIPObjective(num_augmentations=8, cfg_scale=1.2)

# Set prompt
objective.prompt = [
    "a photo of a tiger",
    "a photo of a husky",
    "a photo of a sparrow",
]

# Optionally set a negative prompt
# Note: also need to set cfg_scale > 1 in CLIPObjective if using this!
objective.neg_prompt = "bad, low-res, unnatural"

# Configure test time optimization

In [ ]:
tto_config = TestTimeOptConfig(
    num_iter=301,
    ema_decay=0.98,
    lr=1e-1,
    enable_amp=True,
    reg_weight=0.025,
    #token_noise=1e-3,
    reg_type="seed",
)
tto = TestTimeOpt(tto_config, objective).to(device)

# Load seed images

In [ ]:
# Load seed image
img = torch.cat([
    load_img("ILSVRC2012_val_00008636.png", device),
    load_img("ILSVRC2012_val_00008636.png", device),
    load_img("ILSVRC2012_val_00010240.png", device),
], dim=0)

# Alternatively, initialize directly from given tokens (e.g. randomly
# sampled), but this is disabled when setting `seed_tokens = None`.
seed_tokens = None

# Run Optimization

In [ ]:
print("Seed")
display_image(*img)

# Run optimization
torch.manual_seed(0)
img_opt = tto(
    seed=img if seed_tokens is None else None,
    seed_tokens=seed_tokens,
    callback=opt_callback
)

In [ ]:
def encode_tokens_titok(model, img, quantize=False):
    with torch.no_grad():
        tok = model.encoder(pixel_values=img, latent_tokens=model.latent_tokens)
        
        if quantize:
            if model.quantize_mode == "vq":
                tok, indices, loss = model.quantize(tok)
            else:
                from titok.modeling.quantizer import DiagonalGaussianDistribution
                tok = DiagonalGaussianDistribution(tok).mean
        
        return tok


def decode_tokens_titok(model, tokens, quantize=False):
    with torch.no_grad():
        if not quantize:
            print(f"Tokens not yet quantized...")
            if model.quantize_mode == "vq":
                tokens, indices, loss = model.quantize(tokens)
            else:
                from titok.modeling.quantizer import DiagonalGaussianDistribution
                tokens = DiagonalGaussianDistribution(tokens).mean
        
        img = model.decode(tokens)
        return img


def swap_token_titok(tokens1, tokens2, token_idx):
    tokens1[:, :, :, token_idx], tokens2[:, :, :, token_idx] = tokens2[:, :, :, token_idx], tokens1[:, :, :, token_idx]
    return tokens1

In [ ]:
quantize_tokens = True

img1 = load_img("ILSVRC2012_val_00008636.png", device)
img2 = load_img("ILSVRC2012_val_00010240.png", device)

tokens1 = encode_tokens_titok(tto.titok, img1, quantize=quantize_tokens)
tokens2 = encode_tokens_titok(tto.titok, img2, quantize=quantize_tokens)

print(f"Tokens shape: {tokens1.shape}")

print("Original Image 1:")
display_image(img1)
print("Original Image 2:")
display_image(img2)

recon1 = decode_tokens_titok(tto.titok, tokens1, quantize=quantize_tokens)
print("Reconstructed Image 1:")
display_image(recon1)

# TiTok Token Swapping Experiment

This section demonstrates token swapping between two images using the TiTok model.

In [ ]:
import sys
sys.path.insert(0, "..")
from titok.modeling.titok import TiTok
from omegaconf import OmegaConf

# Load TiTok model
def load_titok_model(checkpoint_path, device):
    config = OmegaConf.load(f"{checkpoint_path}/config.json")
    model = TiTok(config)
    
    # Load weights
    checkpoint = torch.load(f"{checkpoint_path}/pytorch_model.bin", map_location=device)
    model.load_state_dict(checkpoint, strict=True)
    model.eval()
    model.to(device)
    
    for param in model.parameters():
        param.requires_grad = False
    
    return model

In [ ]:
def encode_tokens_titok(model, img, quantize=False):
    with torch.no_grad():
        # Encode using TiTok encoder
        z = model.encoder(pixel_values=img, latent_tokens=model.latent_tokens)
        
        if quantize:
            if model.quantize_mode == "vq":
                # Vector quantization
                z_quantized, result_dict = model.quantize(z)
                return z_quantized
            else:
                # VAE mode
                from titok.modeling.quantizer.quantizer import DiagonalGaussianDistribution
                posteriors = DiagonalGaussianDistribution(z)
                z_quantized = posteriors.sample()
                return z_quantized
        
        return z


def decode_tokens_titok(model, tokens, quantize=False):
    with torch.no_grad():
        if not quantize:
            print(f"Tokens not yet quantized...")
            if model.quantize_mode == "vq":
                # Vector quantization
                tokens_quantized, result_dict = model.quantize(tokens)
                tokens = tokens_quantized
            else:
                # VAE mode
                from titok.modeling.quantizer.quantizer import DiagonalGaussianDistribution
                posteriors = DiagonalGaussianDistribution(tokens)
                tokens = posteriors.sample()
        
        # Decode using TiTok decoder
        img = model.decode(tokens)
        return img


def swap_token_titok(tokens1, tokens2, token_idx):
    tokens1_copy = tokens1.clone()
    tokens2_copy = tokens2.clone()
    
    # Swap the token at the specified index
    # Assuming tokens shape is [batch, channels, height, width]
    tokens1_copy[:, token_idx, :, :], tokens2_copy[:, token_idx, :, :] = \
        tokens2_copy[:, token_idx, :, :].clone(), tokens1_copy[:, token_idx, :, :].clone()
    
    return tokens1_copy

In [ ]:
# Load TiTok model - specify your checkpoint path here
checkpoint_path = "path/to/titok/checkpoint"  # Update this path
titok_model = load_titok_model(checkpoint_path, device)
print(f"TiTok model loaded successfully")
print(f"Quantize mode: {titok_model.quantize_mode}")
print(f"Number of latent tokens: {titok_model.num_latent_tokens}")

In [ ]:
# Set whether to quantize tokens
quantize_tokens = True

# Load two images for token swapping
img1 = load_img("ILSVRC2012_val_00008636.png", device)
img2 = load_img("ILSVRC2012_val_00010240.png", device)

# Encode images to tokens
tokens1 = encode_tokens_titok(titok_model, img1, quantize=quantize_tokens)
tokens2 = encode_tokens_titok(titok_model, img2, quantize=quantize_tokens)

print(f"Tokens shape: {tokens1.shape}")
print(f"Token dimensions: {tokens1.shape}")

# Display original images
print("="*60)
print("Original Image 1:")
display_image(img1)
print("="*60)
print("Original Image 2:")
display_image(img2)

In [ ]:
# Test reconstruction of Image 1
recon1 = decode_tokens_titok(titok_model, tokens1, quantize=quantize_tokens)
print("="*60)
print("Reconstructed Image 1:")
display_image(recon1)

## Token Swapping Loop

Now we'll swap each token one at a time and visualize the results.

In [ ]:
import imageio
from pathlib import Path

def save_img_tensor(tensor, path):
    tensor = tensor.squeeze().clamp(0., 1.) * 255.
    img = Image.fromarray(tensor.permute(1, 2, 0).cpu().to(dtype=torch.uint8).numpy())
    img.save(path)
    return img

# Create output directory
output_dir = Path("titok_token_swap_frames")
output_dir.mkdir(exist_ok=True)

# Perform token swapping for each token and collect frames
num_tokens = tokens1.shape[1]  # Assuming shape is [batch, num_tokens, ...]

print(f"Starting token swapping experiment...")
print(f"Total tokens to swap: {num_tokens}")
print("="*60)

frames = []  # List to store all frames for GIF

# Save original images
save_img_tensor(img1, output_dir / "original_img1.png")
save_img_tensor(img2, output_dir / "original_img2.png")

for token_idx in range(num_tokens):
    print(f"Processing token {token_idx}/{num_tokens-1}", end="\r")
    
    # Clone original tokens for this iteration
    tokens1_current = tokens1.clone()
    tokens2_current = tokens2.clone()
    
    # Swap the token
    tokens_swapped = swap_token_titok(tokens1_current, tokens2_current, token_idx)
    
    # Decode the swapped tokens
    img_swapped = decode_tokens_titok(titok_model, tokens_swapped, quantize=quantize_tokens)
    
    # Save frame
    frame_path = output_dir / f"frame_{token_idx:03d}.png"
    img_pil = save_img_tensor(img_swapped, frame_path)
    frames.append(np.array(img_pil))
    
    # Display every 8th frame to avoid cluttering output
    if token_idx % 8 == 0:
        print(f"\nToken {token_idx}/{num_tokens-1}:")
        display_image(img_swapped)

print("\n" + "="*60)
print("Token swapping completed! Creating GIF...")

# Create GIF animation
gif_path = output_dir / "token_swap_animation.gif"
imageio.mimsave(gif_path, frames, duration=0.1, loop=0)

print(f"GIF saved to: {gif_path}")
print(f"Total frames: {len(frames)}")
print(f"All frames saved to: {output_dir}")
print("="*60)

# Display the first, middle, and last frame
print("\nFirst frame:")
display_image(img1)
print("\nMiddle frame:")
middle_idx = num_tokens // 2
tokens1_mid = tokens1.clone()
tokens2_mid = tokens2.clone()
tokens_mid = swap_token_titok(tokens1_mid, tokens2_mid, middle_idx)
img_mid = decode_tokens_titok(titok_model, tokens_mid, quantize=quantize_tokens)
display_image(img_mid)
print("\nLast frame:")
tokens1_last = tokens1.clone()
tokens2_last = tokens2.clone()
tokens_last = swap_token_titok(tokens1_last, tokens2_last, num_tokens-1)
img_last = decode_tokens_titok(titok_model, tokens_last, quantize=quantize_tokens)
display_image(img_last)